# Comparing Time-Dependent Solvers ⏱️

POPSIM implements several solvers for time-dependent simulations. This notebook demonstrates their differences in performance and accuracy.

tl;dr:
Use `SIMPLE_EULER` for training with short to moderate segment lengths, and `DIFFRAX_TSIT5` when training over long time horizons or when additional precision is needed.

A list of the Steppers we have implemented is shown below. See the [Diffrax docs](https://docs.kidger.site/diffrax/usage/how-to-choose-a-solver/) for info on the Diffrax solvers.

In [ ]:
import inspect
from IPython.display import Markdown, display
from popsim.simulate import StepperType
display(Markdown(f"```python\n{inspect.getsource(StepperType)}\n```"))

In [ ]:
"""Set up a simple time-dependent module and functions to run simulations with timing."""
import time

import chex
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import xarray as xr
from IPython.display import Markdown, display
from loguru import logger
from tabulate import tabulate

from popsim import TimeDepModule
from popsim.simulate import SimInput, StepperType, make_time_base, simulate

class VanDerPolOscillator(TimeDepModule):
    """Van der Pol oscillator - a nonlinear oscillator with a stable limit cycle.
    This is a classic attractor system that exhibits self-sustained oscillations.
    The parameter mu controls the nonlinearity and damping.
    """

    @chex.dataclass
    class Config:
        mu: float = 1.0  # Nonlinearity parameter (mu > 0 gives limit cycle)

    @chex.dataclass
    class State:
        x: float  # Position
        v: float  # Velocity

    @chex.dataclass
    class Output:
        x_out: float
        v_out: float
        energy: float  # Total energy
        phase: float  # Phase angle in state space

    @chex.dataclass
    class Inputs:
        driving_force: float = 0.0  # External driving force

    config: Config

    def __init__(self, config=None):
        self.config = config or VanDerPolOscillator.Config()

    def __call__(self, state: State, inputs: Inputs) -> tuple[State, Output]:
        # Van der Pol equations:
        # dx/dt = v
        # dv/dt = mu * (1 - x^2) * v - x + driving_force

        x_dot = state.v
        v_dot = self.config.mu * (1.0 - state.x**2) * state.v - state.x + inputs.driving_force

        state_dot = VanDerPolOscillator.State(x=x_dot, v=v_dot)

        # Output
        energy = 0.5 * state.v**2 + 0.5 * state.x**2  # Approximate energy
        phase = jnp.arctan2(state.v, state.x)
        out = VanDerPolOscillator.Output(x_out=state.x, v_out=state.v, energy=energy, phase=phase)

        return state_dot, out

def run_simulation_with_timing(
    module: TimeDepModule,
    sim_input: SimInput,
    stepper_type: StepperType,
    warmup_runs: int = 2,
    timing_runs: int = 3,
) -> tuple[xr.Dataset, float, float]:
    """Run simulation with timing measurements.

    Args:
        module: The module to simulate
        sim_input: Simulation input
        stepper_type: Type of stepper to use
        warmup_runs: Number of warmup runs for JIT compilation
        timing_runs: Number of timing runs to average

    Returns:
        Tuple of (result_dataset, jit_time, mean_execution_time)
    """

    # Warmup run (includes JIT compilation)
    start_jit = time.time()
    result = simulate(module, sim_input, stepper_type=stepper_type, return_xarray=True)
    jax.block_until_ready(result)
    jit_time = time.time() - start_jit

    # Additional warmup runs
    for i in range(warmup_runs - 1):
        result = simulate(module, sim_input, stepper_type=stepper_type, return_xarray=True)
        jax.block_until_ready(result)

    # Timing runs
    execution_times = []
    for i in range(timing_runs):
        start = time.time()
        result = simulate(module, sim_input, stepper_type=stepper_type, return_xarray=True)
        jax.block_until_ready(result)
        exec_time = time.time() - start
        execution_times.append(exec_time)

    mean_execution_time = jnp.mean(jnp.array(execution_times))

    return result, jit_time, float(mean_execution_time)

def compute_differences(datasets: list[xr.Dataset]) -> dict[str, dict[str, xr.DataArray]]:
    """Compute pairwise differences between xarray Datasets.

    Uses the first dataset as the reference and computes differences for all others.

    Args:
        datasets: List of xarray Datasets to compare (length >= 2)

    Returns:
        Dictionary mapping variable names to dictionaries of difference metrics.
        Structure: {var_name: {stepper_idx: {"absolute": ..., "relative": ...}}}
    """
    if len(datasets) < 2:
        raise ValueError("Need at least 2 datasets to compute differences")

    differences = {}
    ds_ref = datasets[0]  # Use first dataset as reference

    # Find common variables across all datasets
    common_vars = set(ds_ref.data_vars)
    for ds in datasets[1:]:
        common_vars = common_vars.intersection(set(ds.data_vars))

    for var in common_vars:
        differences[var] = {}

        # Compute differences for each dataset relative to reference
        for i, ds in enumerate(datasets[1:], start=1):
            # Compute absolute difference using xarray operations
            abs_diff = xr.apply_ufunc(
                lambda x, y: jnp.abs(x - y),
                ds_ref[var],
                ds[var],
                dask="parallelized",
            )

            # Compute relative difference (handle division by zero)
            denominator = xr.apply_ufunc(jnp.abs, ds_ref[var]) + 1e-10  # Add small epsilon
            rel_diff = abs_diff / denominator

            differences[var][i] = {"absolute": abs_diff, "relative": rel_diff}

    return differences

def plot_comparison(
    datasets: list[xr.Dataset],
    stepper_names: list[str],
    differences: dict[str, dict[str, xr.DataArray]],
):
    """Plot comparison of results and differences for multiple steppers.

    Args:
        datasets: List of result datasets from different steppers
        stepper_names: List of stepper names (for labels)
        differences: Computed differences between datasets
        save_path: Path to save the plot
    """
    ds_ref = datasets[0]  # Reference dataset for variable detection

    # Define colors and line styles for plotting
    colors = plt.cm.tab10(range(len(datasets)))
    line_styles = ["-", "--", "-.", ":"] * (len(datasets) // 4 + 1)

    # Automatically detect state and output variables to plot
    state_vars = sorted([v for v in ds_ref.data_vars if v.startswith("state.")])
    output_vars = sorted([v for v in ds_ref.data_vars if v.startswith("output.")])
    input_vars = sorted([v for v in ds_ref.data_vars if v.startswith("inputs.")])

    # Prioritize certain output variables if they exist
    priority_outputs = ["output.energy", "output.phase"]
    other_outputs = [v for v in output_vars if v not in priority_outputs]
    plot_output_vars = [v for v in priority_outputs if v in output_vars] + other_outputs

    # Select key variables to plot (state + up to 2 output variables)
    plot_vars = state_vars + plot_output_vars[:2]

    # Add 1 extra row for the input variable if it exists
    n_vars = len(plot_vars) + (1 if input_vars else 0)
    _fig, axes = plt.subplots(n_vars, 3, figsize=(15, 4 * n_vars))

    if n_vars == 1:
        axes = axes.reshape(1, -1)

    for i, var in enumerate(plot_vars):
        if var not in ds_ref:
            continue

        time = ds_ref["time"].values

        # Plot trajectories for all steppers
        for j, (ds, name) in enumerate(zip(datasets, stepper_names)):
            if var in ds:
                axes[i, 0].plot(
                    time, ds[var].values,
                    label=name,
                    linewidth=2,
                    color=colors[j],
                    linestyle=line_styles[j]
                )
        axes[i, 0].set_xlabel("Time")
        axes[i, 0].set_ylabel(var)
        axes[i, 0].set_title(f"{var} - Trajectories")
        axes[i, 0].legend()
        axes[i, 0].grid(True, alpha=0.3)

        # Plot absolute differences (relative to first stepper)
        if var in differences:
            for stepper_idx, diff_data in differences[var].items():
                abs_diff = diff_data["absolute"].values
                axes[i, 1].plot(
                    time, abs_diff,
                    label=f"{stepper_names[stepper_idx]} vs {stepper_names[0]}",
                    linewidth=2,
                    color=colors[stepper_idx],
                    linestyle=line_styles[stepper_idx]
                )
            axes[i, 1].set_xlabel("Time")
            axes[i, 1].set_ylabel("Absolute Difference")
            axes[i, 1].set_title(f"{var} - Absolute Difference")
            axes[i, 1].legend()
            axes[i, 1].grid(True, alpha=0.3)
            axes[i, 1].set_yscale("log")

            # Plot relative differences
            for stepper_idx, diff_data in differences[var].items():
                rel_diff = diff_data["relative"].values
                axes[i, 2].plot(
                    time, rel_diff,
                    label=f"{stepper_names[stepper_idx]} vs {stepper_names[0]}",
                    linewidth=2,
                    color=colors[stepper_idx],
                    linestyle=line_styles[stepper_idx]
                )
            axes[i, 2].set_xlabel("Time")
            axes[i, 2].set_ylabel("Relative Difference")
            axes[i, 2].set_title(f"{var} - Relative Difference")
            axes[i, 2].legend()
            axes[i, 2].grid(True, alpha=0.3)
            axes[i, 2].set_yscale("log")

    # Add input variable plot in the last row if inputs exist
    if input_vars:
        # Use the first input variable
        input_var = input_vars[0]
        last_row = n_vars - 1
        time = ds_ref["time"].values

        # Plot input trajectory for all steppers (should be identical)
        for j, (ds, name) in enumerate(zip(datasets, stepper_names)):
            if input_var in ds:
                axes[last_row, 0].plot(
                    time, ds[input_var].values,
                    label=name,
                    linewidth=2,
                    color=colors[j],
                    linestyle=line_styles[j]
                )
        axes[last_row, 0].set_xlabel("Time")
        axes[last_row, 0].set_ylabel(input_var)
        axes[last_row, 0].set_title(f"{input_var} - Time-Dependent Input")
        axes[last_row, 0].legend()
        axes[last_row, 0].grid(True, alpha=0.3)

        # Plot absolute differences for inputs
        if input_var in differences:
            for stepper_idx, diff_data in differences[input_var].items():
                abs_diff = diff_data["absolute"].values
                axes[last_row, 1].plot(
                    time, abs_diff,
                    label=f"{stepper_names[stepper_idx]} vs {stepper_names[0]}",
                    linewidth=2,
                    color=colors[stepper_idx],
                    linestyle=line_styles[stepper_idx]
                )
            axes[last_row, 1].set_xlabel("Time")
            axes[last_row, 1].set_ylabel("Absolute Difference")
            axes[last_row, 1].set_title(f"{input_var} - Absolute Difference")
            axes[last_row, 1].legend()
            axes[last_row, 1].grid(True, alpha=0.3)

            # Plot relative differences for inputs
            for stepper_idx, diff_data in differences[input_var].items():
                rel_diff = diff_data["relative"].values
                axes[last_row, 2].plot(
                    time, rel_diff,
                    label=f"{stepper_names[stepper_idx]} vs {stepper_names[0]}",
                    linewidth=2,
                    color=colors[stepper_idx],
                    linestyle=line_styles[stepper_idx]
                )
            axes[last_row, 2].set_xlabel("Time")
            axes[last_row, 2].set_ylabel("Relative Difference")
            axes[last_row, 2].set_title(f"{input_var} - Relative Difference")
            axes[last_row, 2].legend()
            axes[last_row, 2].grid(True, alpha=0.3)

    plt.tight_layout()

def print_summary_table(
    stepper_names: list[str],
    jit_times: list[float],
    exec_times: list[float],
    differences: dict[str, dict[str, xr.DataArray]],
):
    """Display summary table of timing and difference metrics in Jupyter-friendly format.

    Args:
        stepper_names: List of stepper names
        jit_times: List of JIT compilation times for each stepper
        exec_times: List of execution times for each stepper
        differences: Computed differences between datasets
    """
    # Title
    display(Markdown("## Stepper Comparison Summary"))

    # Timing comparison
    display(Markdown("### Timing Comparison"))
    timing_data = [["Metric"] + stepper_names + ["Fastest"]]

    # JIT compilation times
    jit_row = ["JIT Compilation Time [s]"] + [f"{t:.4f}" for t in jit_times]
    fastest_jit_idx = jnp.argmin(jnp.array(jit_times))
    jit_row.append(stepper_names[fastest_jit_idx])
    timing_data.append(jit_row)

    # Execution times
    exec_row = ["Mean Execution Time [s]"] + [f"{t:.4f}" for t in exec_times]
    fastest_exec_idx = jnp.argmin(jnp.array(exec_times))
    exec_row.append(stepper_names[fastest_exec_idx])
    timing_data.append(exec_row)

    # Speedup relative to first stepper
    speedup_row = [f"Speedup vs {stepper_names[0]}"] + ["1.00x"]
    for i in range(1, len(exec_times)):
        speedup = exec_times[0] / exec_times[i]
        speedup_row.append(f"{speedup:.2f}x")
    speedup_row.append("-")
    timing_data.append(speedup_row)

    # Display timing table as markdown
    timing_markdown = tabulate(timing_data, headers="firstrow", tablefmt="pipe")
    display(Markdown(timing_markdown))

    # Difference statistics (relative to first stepper)
    if differences:
        display(Markdown(f"### Difference Statistics (relative to {stepper_names[0]})"))

        for var in sorted(differences.keys()):
            if not differences[var]:  # Skip if no differences for this variable
                continue

            display(Markdown(f"#### Variable: `{var}`"))
            diff_data = [["Stepper", "Max Absolute Diff", "Mean Absolute Diff", "Max Relative Diff (%)", "Mean Relative Diff (%)"]]

            for stepper_idx in sorted(differences[var].keys()):
                diff = differences[var][stepper_idx]
                abs_diff = diff["absolute"]
                rel_diff = diff["relative"] * 100  # Convert to percentage

                diff_data.append(
                    [
                        f"{stepper_names[stepper_idx]} vs {stepper_names[0]}",
                        f"{float(abs_diff.max()):.6e}",
                        f"{float(abs_diff.mean()):.6e}",
                        f"{float(rel_diff.max()):.6e}",
                        f"{float(rel_diff.mean()):.6e}",
                    ]
                )

            # Display difference table as markdown
            diff_markdown = tabulate(diff_data, headers="firstrow", tablefmt="pipe")
            display(Markdown(diff_markdown))

def compare_steppers(
    module: TimeDepModule,
    sim_input: SimInput,
    stepper_types: list[StepperType],
):
    """Run comparison for a single module with multiple steppers.

    Args:
        module: The module to simulate
        sim_input: Simulation input
        stepper_types: List of stepper types to compare
    """
    datasets = []
    jit_times = []
    exec_times = []
    stepper_names = []

    # Run simulations for each stepper type
    for stepper_type in stepper_types:
        stepper_name = stepper_type.name
        stepper_names.append(stepper_name)

        # Run simulation with timing
        ds, jit_time, exec_time = run_simulation_with_timing(
            module, sim_input, stepper_type, warmup_runs=2, timing_runs=3
        )

        datasets.append(ds)
        jit_times.append(jit_time)
        exec_times.append(exec_time)

        # Clear JAX cache before running next stepper
        jax.clear_caches()

    differences = compute_differences(datasets)
    print_summary_table(stepper_names, jit_times, exec_times, differences)
    plot_comparison(datasets, stepper_names, differences)

dt = 0.01
t_final = 50.0
time_base = make_time_base(0.0, t_final, dt)

config = VanDerPolOscillator.Config(mu=2.0)
module = VanDerPolOscillator(config)
initial_state = VanDerPolOscillator.State(x=0.1, v=0.1)

# Small time-varying driving force
time_dependent_driving = {
    0.0: 0.0,
    10.0: 0.1,
    30.0: 0.1,
    40.0: 0.0,
    50.0: 0.0,
}

inputs = VanDerPolOscillator.Inputs(driving_force=time_dependent_driving)
sim_input = SimInput(time=time_base, initial_state=initial_state, inputs=inputs)

# 1. Comparison between `SIMPLE_EULER` and `DIFFRAX_EULER`

The `DIFFRAX_EULER` method and the `SIMPLE_EULER` method produce near-identical results, but the `SIMPLE_EULER` method compiles ~30x faster and runs ~3x faster from our testing.

In [ ]:
stepper_types = [StepperType.DIFFRAX_EULER, StepperType.SIMPLE_EULER]
compare_steppers(module, sim_input, stepper_types)

# 2. Comparison between DIFFRAX solvers

In [ ]:
stepper_types = [StepperType.DIFFRAX_TSIT5, StepperType.DIFFRAX_DOPRI5, StepperType.SIMPLE_EULER]
compare_steppers(module, sim_input, stepper_types)